## Downloads Documents

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import tabula
import pandas as pd

# Base URL of the website
base_url = 'https://rbi.org.in/Scripts/AnnualPublications.aspx?head=State+Finances+%3a+A+Study+of+Budgets'

# Set headers to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Define the path where you want to store the files
download_folder = r'C:/Users/chandresh.kanyal/Downloads'  # Change this to your desired folder path

# Ensure the directory exists
if not os.path.exists(download_folder):
    os.makedirs(download_folder)

# Send a GET request to the website
response = requests.get(base_url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the table that contains the row for Statement 21
    table = soup.find('table')  # You may need to adjust this if there are multiple tables

    # Iterate through each row in the table to find the relevant one
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) > 0 and 'Appendix Table 3: Development and Non-Development Expenditure: Aggregate' in cells[0].text:
            # This is the correct row
            print("Found the row for Statement 21")

            # Look for the download links (Excel and PDF)
            excel_link = None
            pdf_link = None

            # Iterate through the cells to find the download links (Excel and PDF)
            for cell in cells:
                # Find all anchor tags with the 'href' attribute (links)
                links = cell.find_all('a', href=True)
                for link in links:
                    href = link['href']
                    print(f"Found link: {href}")  # Debugging: print each link found
                    
                    # Check if the link is an Excel or PDF file
                    if '.xlsx' in href:
                        excel_link = urljoin(base_url, href)  # Resolve relative URL
                    elif '.pdf' in href:
                        pdf_link = urljoin(base_url, href)  # Resolve relative URL

            # Check and download the Excel file if the link is found
            if excel_link:
                print(f"Found Excel file link: {excel_link}")
                try:
                    excel_response = requests.get(excel_link, headers=headers, allow_redirects=True)
                    print(f"Excel response status: {excel_response.status_code}")  # Debugging response status
                    excel_response.raise_for_status()  # Raise an error for bad status codes
                    excel_file_path = os.path.join(download_folder, 'Statement_21_Market_Borrowings.xlsx')
                    with open(excel_file_path, 'wb') as f:
                        f.write(excel_response.content)
                    print(f"Excel file downloaded successfully to: {excel_file_path}")
                except requests.exceptions.RequestException as e:
                    print(f"Error downloading Excel file: {e}")

            # Check and download the PDF file if the link is found
            if pdf_link:
                print(f"Found PDF file link: {pdf_link}")
                try:
                    pdf_response = requests.get(pdf_link, headers=headers, allow_redirects=True)
                    print(f"PDF response status: {pdf_response.status_code}")  # Debugging response status
                    pdf_response.raise_for_status()  # Raise an error for bad status codes
                    pdf_file_path = os.path.join(download_folder, 'Statement_21_Market_Borrowings.pdf')
                    with open(pdf_file_path, 'wb') as f:
                        f.write(pdf_response.content)
                    print(f"PDF file downloaded successfully to: {pdf_file_path}")
                except requests.exceptions.RequestException as e:
                    print(f"Error downloading PDF file: {e}")
            
            # Break the loop after downloading
            break
else:
    print(f"Failed to retrieve the webpage. Status code: {response.status_code}")


Found the row for Statement 21
Found link: PublicationsView.aspx?id=22255
Found link: https://rbidocs.rbi.org.in/rdocs/Publications/DOCs/03APP11122023BF883F4BDF6547069678909F70EC5C74.XLSX
Found link: https://rbidocs.rbi.org.in/rdocs/Publications/PDFs/03APP11122023BF883F4BDF6547069678909F70EC5C74.PDF


In [3]:
# Path to your PDF file
pdf_file_path = r"C:/Users/chandresh.kanyal/Desktop/Presentation/03APP11122023BF883F4BDF6547069678909F70EC5C74.PDF"
output_excel_path = r"C:/Users/chandresh.kanyal/Desktop/Presentation/Appendix.xlsx"

# Extract tables from the PDF
# `pages='all'` to extract tables from all pages, you can specify a specific page as well
tables = tabula.read_pdf(pdf_file_path, pages='all', multiple_tables=True)

# Check if tables were extracted and save them to Excel
if tables:
    # Combine all tables (if multiple tables were extracted) into one DataFrame
    combined_df = pd.concat(tables, ignore_index=True)

    # Save to Excel
    combined_df.to_excel(output_excel_path, index=False)
    print(f"PDF data extracted and saved to {output_excel_path}")
else:
    print("No tables found in the PDF.")


Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'


PDF data extracted and saved to C:/Users/chandresh.kanyal/Downloads/Appendix.xlsx
